# ETL Superstore → Star Schema

Tiga file sumber dibersihkan, digabungkan, lalu dimuat ke **star schema** di PostgreSQL.
**Extract, Transform, dan Load dikerjakan dengan Spark. Database dan tabel dibuat dengan SQL** (file di folder `sql/`).

| Langkah | Alat | Di mana |
| --- | --- | --- |
| 0. Buat database & tabel | SQL | pgAdmin: `sql/01_create_database.sql`, `sql/02_create_tables.sql` |
| 1. Extract: baca 3 file CSV | Spark | notebook ini |
| 2. Transform: bersihkan, join, buat dimension & fact | Spark | notebook ini |
| 3. Load: isi tabel | Spark | notebook ini |
| 4. Cek hasil | SQL | pgAdmin: `sql/03_check.sql` |

### Dari 3 file sumber menjadi star schema

```
superstore_customers.csv ──►  dim_customer
superstore_products.csv  ──►  dim_product
superstore_orders.csv    ──►  dim_location, dim_date, fact_sales

                 dim_customer
                      │
   dim_product ── fact_sales ── dim_location
                      │
                   dim_date
```

- **Tabel dimension** (`dim_`): menjelaskan *siapa / apa / di mana / kapan*. Satu baris per customer, produk, lokasi, tanggal.
- **Tabel fact** (`fact_`): kejadian yang diukur (sales, quantity, profit), berisi ID yang menunjuk ke dimension.

## 0. Buat database & tabel (SQL, di pgAdmin)

Lakukan **sekali, sebelum menjalankan notebook ini**:

1. pgAdmin → klik kanan server → *Query Tool* (terhubung ke database `postgres`) → buka `sql/01_create_database.sql` → jalankan. Database `dwh` terbentuk.
2. Klik kanan *Databases* → *Refresh* → pilih `dwh` → *Query Tool* → buka `sql/02_create_tables.sql` → jalankan. Lima tabel kosong terbentuk, lengkap dengan primary key dan foreign key.

Nama kolom di `02_create_tables.sql` harus sama dengan DataFrame yang kita buat di bawah.

### SETUP

In [ ]:
import os
import sys
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

load_dotenv('../.env')                          # baca PG_HOST, PG_PASSWORD, dst. dari file .env
os.environ['PYSPARK_PYTHON'] = sys.executable   # Spark memakai Python dari env conda ini

java_home = os.path.join(sys.prefix, 'Library' if os.name == 'nt' else os.path.join('lib', 'jvm'))
if 'JAVA_HOME' not in os.environ and os.path.isdir(java_home):
    os.environ['JAVA_HOME'] = java_home          # pakai Java dari env conda (penting saat dijalankan di VS Code)

spark = (SparkSession.builder
         .appName("superstore-etl")
         .master("local[2]")                           # jalan di laptop, pakai 2 core
         .config("spark.driver.memory", "1g")          # RAM maksimal 1 GB
         .config("spark.sql.shuffle.partitions", "4")  # saat groupBy/join, data dibagi jadi 4 bagian
         .config("spark.jars.packages", "org.postgresql:postgresql:42.7.14")  # driver PostgreSQL, di-download otomatis
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # sembunyikan pesan warning Spark

Penjelasan setting di `SparkSession.builder`:

| Setting | Arti |
| --- | --- |
| `master("local[2]")` | Spark jalan di laptop sendiri dengan 2 core. Di cluster, isinya alamat cluster |
| `spark.driver.memory` | RAM maksimal yang boleh dipakai Spark |
| `spark.sql.shuffle.partitions` | Jumlah bagian data saat `groupBy`/`join`. Default 200, terlalu banyak untuk data kecil |
| `PYSPARK_PYTHON` | Memastikan Spark memakai Python dari env conda yang aktif |
| `JAVA_HOME` | Lokasi Java dari env conda. Diisi otomatis kalau kosong, misalnya saat notebook dijalankan dari VS Code |
| `spark.jars.packages` | Driver JDBC PostgreSQL. Di-download otomatis saat pertama kali jalan (butuh internet, ±1 MB) |
| `load_dotenv('../.env')` | Membaca isi file `.env` (host, user, password PostgreSQL) |

## 1. Extract

In [ ]:
def read_csv(file_name):
    return spark.read.csv(f'../data/{file_name}', header=True, inferSchema=True, escape='"')

orders_raw = read_csv('superstore_orders.csv')
customers_raw = read_csv('superstore_customers.csv')
products_raw = read_csv('superstore_products.csv')

print('orders:', orders_raw.count(), '| customers:', customers_raw.count(), '| products:', products_raw.count())

## 2. Transform

### Step 1. Ubah nama kolom ke snake_case

`Customer ID` → `customer_id`. Nama kolom tanpa spasi lebih mudah dipakai di Spark, dan harus sama dengan tabel SQL.
pandas: `df.columns = [...]` → Spark: `df.toDF(*nama_baru)`

In [ ]:
def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(orders_raw)
customers = to_snake_case(customers_raw)
products = to_snake_case(products_raw)

print(orders.columns)
print(customers.columns)
print(products.columns)

### Step 2. Cek kualitas data

**Baris duplikat** → pandas: `df.duplicated().sum()`

In [ ]:
for name, df in [('orders', orders), ('customers', customers), ('products', products)]:
    print(name, df.count() - df.dropDuplicates().count())

**Nilai kosong di orders** → pandas: `df.isna().sum()`

In [ ]:
for c in orders.columns:
    n = orders.filter(F.col(c).isNull()).count()
    if n > 0:
        print(c, n)

**Teks tidak konsisten**

In [ ]:
customers.groupBy('segment').count().show()
products.groupBy('category').count().show()
orders.select('region').distinct().show()

**Product ID yang punya lebih dari 1 nama** di file products:

In [ ]:
products.groupBy('product_id').agg(F.countDistinct('product_name').alias('names')).filter('names > 1').show(3)

**Postal code yang kehilangan angka 0 di depan** (`6824` seharusnya `06824`) dan **tanggal yang masih berupa teks**:

In [ ]:
orders.filter(F.length(F.col('postal_code').cast('string')) < 5).select('city', 'state', 'postal_code').show(3)
orders.select('order_date', 'ship_date').printSchema()

### Step 3. Bersihkan setiap tabel

| Masalah | Solusi | pandas |
| --- | --- | --- |
| baris duplikat | `dropDuplicates()` | `drop_duplicates()` |
| spasi berlebih & huruf besar/kecil campur | `F.initcap(F.trim(col))` | `.str.strip().str.title()` |
| 1 ID, beberapa nama | `dropDuplicates(['product_id'])` menyisakan 1 baris per ID | `drop_duplicates('Product ID')` |
| postal code tanpa angka 0 di depan | `F.lpad(col, 5, '0')` | `.str.zfill(5)` |
| postal code kosong | `fillna({'postal_code': 'Unknown'})` | `fillna(...)` |
| tanggal berupa teks | `F.to_date(col, 'M/d/yyyy')` | `pd.to_datetime(col, format='%m/%d/%Y')` |
| sales kosong | `dropna(subset=['sales'])` | `dropna(subset=[...])` |

In [ ]:
customers_clean = (customers
    .dropDuplicates()
    .withColumn('customer_name', F.trim('customer_name'))
    .withColumn('segment', F.initcap(F.trim('segment')))
    .dropDuplicates(['customer_id']))

products_clean = (products
    .dropDuplicates()
    .withColumn('category', F.initcap(F.trim('category')))
    .dropDuplicates(['product_id']))

orders_clean = (orders
    .dropDuplicates()
    .withColumn('region', F.initcap(F.trim('region')))
    .withColumn('postal_code', F.lpad(F.col('postal_code').cast('string'), 5, '0'))
    .fillna({'postal_code': 'Unknown'})
    .withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))
    .withColumn('ship_date', F.to_date('ship_date', 'M/d/yyyy'))
    .dropna(subset=['sales']))

print('customers:', customers.count(), '->', customers_clean.count())
print('products :', products.count(), '->', products_clean.count())
print('orders   :', orders.count(), '->', orders_clean.count())
customers_clean.groupBy('segment').count().show()

### Step 4. Gabungkan sumber data (join)

Sebelum membuat tabel fact, cek dulu apakah setiap order menunjuk ke customer dan produk yang ada.
`left_anti` mengembalikan order yang **tidak punya** pasangan.

In [ ]:
missing_customer = orders_clean.join(customers_clean, on='customer_id', how='left_anti')
missing_product = orders_clean.join(products_clean, on='product_id', how='left_anti')

print('order tanpa customer:', missing_customer.count())
print('order tanpa produk  :', missing_product.count())
missing_product.select('row_id', 'order_id', 'product_id', 'sales').show()

Ada 2 order yang menunjuk ke produk yang tidak ada di file products. Foreign key di `fact_sales` akan menolaknya,
jadi kita hanya menyimpan order yang produknya dikenal dengan **inner join** (hanya baris yang cocok yang disimpan).
Di proyek nyata, baris seperti ini dilaporkan ke pemilik data.

In [ ]:
orders_valid = orders_clean.join(products_clean.select('product_id'), on='product_id', how='inner')
print('orders:', orders_clean.count(), '->', orders_valid.count())

**Gabungkan semua sumber** menjadi satu tabel, seperti export aslinya, untuk melihat datanya bersama-sama.
Jumlah baris harus tetap sama dengan `orders_valid`; kalau bertambah, masih ada ID duplikat di salah satu sumber.

In [ ]:
sales_full = (orders_valid
    .join(customers_clean, on='customer_id', how='left')
    .join(products_clean, on='product_id', how='left'))

print('baris:', sales_full.count())
sales_full.groupBy('segment', 'category').agg(F.round(F.sum('sales'), 2).alias('total_sales')).orderBy('segment', 'category').show()

### Step 5. Buat tabel dimension

`dim_customer` dan `dim_product` diambil langsung dari file sumber yang sudah bersih: satu baris per ID.

In [ ]:
dim_customer = customers_clean.select('customer_id', 'customer_name', 'segment')
dim_product = products_clean.select('product_id', 'product_name', 'category', 'sub_category')

dim_customer.show(3)
dim_product.show(3, truncate=False)

Lokasi tidak punya ID di file sumber, jadi kita buat sendiri: ambil lokasi yang unik, urutkan, lalu beri nomor 1, 2, 3, ...

In [ ]:
location_cols = ['country_region', 'region', 'state', 'city', 'postal_code']

dim_location = (orders_valid.select(location_cols).dropDuplicates()
                .withColumn('location_id', F.row_number().over(Window.orderBy('state', 'city', 'postal_code'))))
dim_location.show(3)

Dimension tanggal berisi satu baris per tanggal, dengan bagian-bagian yang berguna untuk laporan.
`date_id` adalah tanggal dalam bentuk angka, misalnya `20201108`.

In [ ]:
dates = (orders_valid.select(F.col('order_date').alias('full_date'))
         .union(orders_valid.select(F.col('ship_date').alias('full_date')))
         .distinct())

dim_date = (dates
    .withColumn('date_id', F.date_format('full_date', 'yyyyMMdd').cast('int'))
    .withColumn('year', F.year('full_date'))
    .withColumn('quarter', F.quarter('full_date'))
    .withColumn('month', F.month('full_date'))
    .withColumn('month_name', F.date_format('full_date', 'MMMM'))
    .withColumn('day_name', F.date_format('full_date', 'EEEE')))
dim_date.orderBy('full_date').show(3)

### Step 6. Buat tabel fact

Tabel fact menyimpan angka (`sales`, `quantity`, `discount`, `profit`) dan hanya **ID** dari setiap dimension.
`location_id` diambil dengan join balik ke `dim_location`.

In [ ]:
fact_sales = (orders_valid
    .join(dim_location, on=location_cols, how='left')
    .withColumn('order_date_id', F.date_format('order_date', 'yyyyMMdd').cast('int'))
    .withColumn('ship_date_id', F.date_format('ship_date', 'yyyyMMdd').cast('int'))
    .select('row_id', 'order_id', 'order_date_id', 'ship_date_id', 'customer_id', 'product_id',
            'location_id', 'ship_mode', 'sales', 'quantity', 'discount', 'profit'))
fact_sales.show(3)

**Cek:** satu baris fact untuk setiap order valid, dan tidak ada ID yang kosong.

In [ ]:
print('order valid:', orders_valid.count(), '| baris fact:', fact_sales.count())
print('location_id kosong:', fact_sales.filter(F.col('location_id').isNull()).count())

for name, table in [('dim_customer', dim_customer), ('dim_product', dim_product),
                    ('dim_location', dim_location), ('dim_date', dim_date), ('fact_sales', fact_sales)]:
    print(name, table.count())

## 3. Load ke PostgreSQL

DataFrame dimasukkan ke tabel yang sudah kita buat dengan SQL di langkah 0, di **PostgreSQL laptop**.

Notebook ini berjalan langsung di laptop, jadi host-nya `localhost`, sama seperti di pgAdmin.
Host, user, dan password diambil dari file `.env` (dibaca dengan `load_dotenv` di SETUP), jadi password **tidak pernah ditulis di notebook**.

In [ ]:
missing = [k for k in ('PG_HOST', 'PG_PORT', 'PG_DB', 'PG_USER', 'PG_PASSWORD') if not os.getenv(k)]
if missing:
    raise SystemExit(f'Belum diisi di file .env: {missing}. Salin .env.example menjadi .env, isi, lalu restart kernel (Kernel → Restart Kernel).')

jdbc_url = f"jdbc:postgresql://{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['PG_DB']}"
jdbc_props = {'user': os.environ['PG_USER'], 'password': os.environ['PG_PASSWORD'], 'driver': 'org.postgresql.Driver'}
print(jdbc_url)

**Cek sebelum load**: koneksi berhasil, tabel dari `02_create_tables.sql` sudah ada, dan tabelnya masih kosong
(load dua kali akan membuat key duplikat).

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(host=os.environ['PG_HOST'], port=os.environ['PG_PORT'], dbname=os.environ['PG_DB'],
                            user=os.environ['PG_USER'], password=os.environ['PG_PASSWORD'], connect_timeout=5)
except psycopg2.OperationalError as e:
    msg = str(e)
    if 'does not exist' in msg:
        raise SystemExit(f"Database {os.environ['PG_DB']} belum ada. Jalankan sql/01_create_database.sql di pgAdmin dulu.")
    if 'password authentication failed' in msg:
        raise SystemExit('Password salah. PG_USER / PG_PASSWORD di .env harus sama dengan pgAdmin.')
    if 'Connection refused' in msg or 'timeout' in msg:
        raise SystemExit('PostgreSQL di laptop tidak bisa dihubungi. Pastikan PostgreSQL berjalan (pgAdmin bisa connect).')
    raise

with conn.cursor() as cur:
    cur.execute("SELECT to_regclass('fact_sales')")
    if cur.fetchone()[0] is None:
        raise SystemExit('Tabel fact_sales belum ada. Jalankan sql/02_create_tables.sql di pgAdmin dulu.')
    cur.execute('SELECT COUNT(*) FROM fact_sales')
    if cur.fetchone()[0] > 0:
        raise SystemExit('Tabel sudah berisi data. Jalankan ulang sql/02_create_tables.sql untuk mengosongkan, lalu jalankan cell ini lagi.')
conn.close()
print('Terhubung. Tabel sudah ada dan masih kosong.')

**Masukkan data.** Dimension dulu, baru fact, karena tabel fact punya foreign key ke tabel dimension.

`mode='append'` menambahkan baris ke tabel yang sudah ada, sehingga desain dari `02_create_tables.sql` tetap terjaga.
(`mode='overwrite'` akan menghapus tabel kita dan Spark membuat tabel baru tanpa primary key & foreign key.)

In [ ]:
tables = [('dim_customer', dim_customer), ('dim_product', dim_product), ('dim_location', dim_location),
          ('dim_date', dim_date), ('fact_sales', fact_sales)]

for name, table in tables:
    table.write.jdbc(jdbc_url, name, mode='append', properties=jdbc_props)
    print('masuk:', name)

## 4. Cek hasil

Baca kembali dari PostgreSQL dengan query yang menggabungkan tabel fact dengan dua dimension.
Query lainnya ada di `sql/03_check.sql` (jalankan di pgAdmin).

In [ ]:
query = '''
(SELECT l.region, p.category, ROUND(SUM(f.sales), 2)::float AS total_sales
 FROM fact_sales f
 JOIN dim_location l ON f.location_id = l.location_id
 JOIN dim_product  p ON f.product_id  = p.product_id
 GROUP BY l.region, p.category
 ORDER BY l.region, p.category) AS t
'''
spark.read.jdbc(jdbc_url, query, properties=jdbc_props).show()

Selesai. Hentikan Spark supaya memori laptop dilepas.

In [ ]:
spark.stop()